# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **Freestyle submission, V2 edition.** This notebook belongs to my final, freestyle capstone: *which ranking pages are losing clicks to AI-answer search before it shows, and which should an editor protect first?* The pipeline was rebuilt as **V2**: the whole process was written down and hashed in `work/V2_SPEC.md` before any V2 code ran. V1 stays in the repository as the first process, and every V1 number shown here is labelled V1. My earlier Lane 2 (refresh) experiments are kept for the record in `work/archive/` and are not part of this submission.

**What this card asks:** two signal checks with bucket tables, n and a verdict; ONE rule with a score, one reason code
and an action label, written to `work/outputs/baseline_action_score.csv`; and a review of the top 20. **The principle it
serves:** the solution hierarchy. A transparent rule comes first, and any model must beat it on the same rows.

In [1]:
# Setup: works locally and in Colab. Reads committed receipts; the warehouse is touched only by the opt-in rerun.
import importlib.util, json, os, sys
if any(importlib.util.find_spec(p) is None for p in ["duckdb", "sklearn", "scipy", "matplotlib"]):
    get_ipython().run_line_magic("pip", "install -q duckdb huggingface_hub pandas scikit-learn scipy matplotlib")
here = os.path.abspath(os.getcwd())
ROOT = next((p for p in [here, os.path.dirname(here), os.path.dirname(os.path.dirname(here))]
             if os.path.isdir(os.path.join(p, "work", "scripts"))), None)
if ROOT is None:  # fresh Colab runtime: fetch the code (no data lives in git)
    get_ipython().system("git clone -q https://github.com/Tessa-Saumu/FlyRank-ML-Internship.git")
    ROOT = os.path.abspath("FlyRank-ML-Internship")
sys.path.insert(0, os.path.join(ROOT, "work", "scripts"))
OUT, FIG = os.path.join(ROOT, "work", "outputs"), os.path.join(ROOT, "work", "figures")
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")
from IPython.display import Image, display, Markdown
import wh
RERUN_PIPELINE = False   # opt-in: True recomputes from the pinned snapshot (needs HF_TOKEN). V2 steps then run in a scratch copy, never in the repo.
DATA = wh.data_available(try_download=RERUN_PIPELINE)
print("pinned warehouse snapshot available:", DATA, "| revision", wh.REV[:8], "| RERUN_PIPELINE =", RERUN_PIPELINE)
receipt = lambda name: json.load(open(os.path.join(OUT, name)))
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)

# V2 receipts (aggregates only) and the helpers every notebook shares.
v2 = lambda n: receipt(f"v2_{n}.json")
BASE = ["rule_position_slipping", "mean_reversion_ctr_excess", "client_shock", "persistence_prev_ratio"]
BASE_LABEL = {"rule_position_slipping": "editor's rule", "mean_reversion_ctr_excess": "CTR above peers",
              "client_shock": "client shock", "persistence_prev_ratio": "persistence"}
def v2_months():
    """(month, role, evaluation block) for the four evaluated months. The logistic champion is in every block."""
    vf, cf, ts = v2("validation_folds"), v2("confirmation_apr"), v2("test")
    return [("2026-02", "validation", vf["folds"]["validation_1"]), ("2026-03", "validation", vf["folds"]["validation_2"]),
            ("2026-04", "confirmation", cf["evaluation"]), ("2026-05", "test", ts["evaluation"])]
def champ(ev):
    return ev["models"].get("LOGISTIC") or ev["models"]["SELECTED:LOGISTIC"]
def fold_table():
    """One row per evaluated month: the champion against the four frozen baselines, same rows, same metric."""
    rows = []
    for m, role, ev in v2_months():
        lg, best = champ(ev), ev["best_baseline_by_ap"]
        rows.append({"month": m, "role": role, "base rate": ev["base_rate"], "rule P@50": ev["models"][BASE[0]]["p@50"],
                     "best baseline (by AP)": BASE_LABEL[best], "best baseline AP": ev["models"][best]["ap"],
                     "V2 AP": lg["ap"], "V2 AP 95% CI": lg["ap_ci"], "V2 P@50": lg["p@50"], "V2 P@50 95% CI": lg["p@50_ci"],
                     "P@50, unmeasurable = miss": lg["p@50_unmeasurable_as_miss"],
                     "rule P@50, unmeasurable = miss": ev["models"][BASE[0]]["p@50_unmeasurable_as_miss"],
                     "rule share of click shortfall (top 50)": ev["models"][BASE[0]]["clicks_lost_captured"],
                     "V2 share of click shortfall (top 50)": lg["clicks_lost_captured"]})
    return pd.DataFrame(rows)

pinned warehouse snapshot available: True | revision 50cbf7c3 | RERUN_PIPELINE = False


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Question:** what rule would an editor write today, and which reason codes can it output? **Why it matters (FlyRank
principle: the solution hierarchy):** a transparent rule is the bar any model must clear.

**Rule:** `score = position tier (top 3 = 2, positions 4-6 = 1, 7-10 = 0) + max(0, -CTR change from month -2 to -1)`,
with a deterministic hash tiebreak. It encodes two common intuitions: AI answers sit at the top, so top positions are
most exposed, and a CTR that has started slipping keeps slipping.

**Reason codes it can output:** `TOP3_SLIPPING` (top-3 position and CTR down more than 5% last month), `TOP3_EXPOSED`
(top-3 position, CTR not falling), `TOP10_SLIPPING` (positions 4 to 10, CTR down) and `TOP10_WATCH` (positions 4 to 10, CTR not
falling). The action label is `review now`, `review if budget allows` or `monitor`.

I froze the rule **before** looking at the checks, on purpose. It represents the status-quo heuristic, so it must not be
fitted to the data. The checks then test whether its two intuitions hold.

**Four frozen baselines, unchanged in V2.** V2 compares its learned candidates with the same four rules as V1: this editor's
rule, CTR above peers (a data-informed one-signal rule), client-wide shock and persistence of last period's ratio. The editor's
rule gives exactly the same AP and P@50 in the V1 and V2 receipts for all four months. The only implementation change is the
persistence baseline's fill value (the median over all eligible pages of the scored origin, knowable at decision time, instead of
over the measurable pages), which moves its AP by 0.0011 at most.

**Two signal checks (training-era months, December and January).**

In [2]:
sc = receipt("signal_checks.json")
for k in sc:
    print(k); display(pd.DataFrame(sc[k]).pivot(index=list(sc[k][0])[1], columns="era", values=["n", "erosion_rate"]))

check1_position_tier


n                 erosion_rate              
era  design_dec_jan later_feb_apr design_dec_jan later_feb_apr
tier                                                          
1-3          2917.0        8577.0          0.266         0.282
4-6          5929.0       12808.0          0.279         0.305
7-10         2317.0        5497.0          0.310         0.299

check2_recent_ctr_change


n                 erosion_rate              
era        design_dec_jan later_feb_apr design_dec_jan later_feb_apr
ctr_change                                                          
fell 5-26%         2711.0        5498.0          0.245         0.241
fell >26%          3958.0        8095.0          0.254         0.307
flat               1109.0        3409.0          0.243         0.204
rose 5-35%         1933.0        5097.0          0.325         0.289
rose >35%          1452.0        4783.0          0.399         0.417

**Check 1: are top positions more exposed?** Tied to the position tiers FlyRank's own report uses (CTR by tier: top 3,
positions 4-10). In the design months, top-3 pages eroded at 0.266, positions 4-6 at 0.279 and positions 7-10 at
0.310 (n = 2,917 / 5,929 / 2,317). **Verdict: OPPOSITE.** Over a two-month horizon, top pages
are *less* likely to lose further clicks. The large fall in top-3 CTR in ML-06 is a level shift that has already happened across
the panel, not a page-level warning sign.

**Check 2: does a slipping CTR keep slipping?** Pages whose CTR rose by more than 35% eroded at 0.399. Pages whose
CTR fell by more than 26% eroded at 0.254 (n = 1,452 and 3,958). **Verdict: OPPOSITE.** A recent bump reverts. In the
later months the falling group rises to 0.307 while bumps stay highest (0.417), so the pattern is not fully stable.

**What it means.** Both intuitions in the frozen rule fail the training-era data. I keep the rule as the baseline
anyway, because it is what an editor would plausibly do without a model. That is exactly the comparison the model has
to win. The other three baselines keep the comparison from being only against a weak strawman.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**Question:** what does the rule produce when it ranks every eligible page? **Why it matters:** a ranked queue with one
reason code per row is what an editor can act on. **Method:** the rule scores every July 2026 page, ranks it, assigns the reason
code and writes `work/outputs/baseline_action_score.csv` (row level, gitignored, so it exists only after the V1 playbook script
`erosion_playbook.py` has run on a machine with the snapshot). The rule reads only two features that V2 did not change
(`pos_f`, `ctr_growth`), so the file and `baseline_metrics.json` are the V1 artefacts, still valid. The notebook reads them and
does not rewrite them.

In [3]:
bm = receipt("baseline_metrics.json")
print("rule:", bm["rule"]); print("reason codes in the July 2026 rule queue:", bm["live_queue_reason_codes"])
path = os.path.join(OUT, "baseline_action_score.csv")
if os.path.exists(path):
    bq = pd.read_csv(path, usecols=["rule_score", "reason_code", "action_label"])
    print("rows in work/outputs/baseline_action_score.csv (gitignored):", len(bq), "| distinct rule scores:", bq.rule_score.nunique())
    display(bq.groupby(["reason_code", "action_label"]).size().to_frame("pages"))
else:
    print("baseline_action_score.csv is gitignored and not on this machine; the receipt above holds its summary")

rule: position tier (top 3 = 2, 4-6 = 1, 7-10 = 0) + max(0, -CTR change month -1 vs -2), hash tiebreak
reason codes in the July 2026 rule queue: {'TOP10_SLIPPING': 7508, 'TOP10_WATCH': 5785, 'TOP3_SLIPPING': 540, 'TOP3_EXPOSED': 267}
rows in work/outputs/baseline_action_score.csv (gitignored): 14100 | distinct rule scores: 6732


,,pages
reason_code,action_label,
TOP10_SLIPPING,review now,7508
TOP10_WATCH,monitor,5785
TOP3_EXPOSED,review if budget allows,267
TOP3_SLIPPING,review now,540


**The rule against the V2 folds.** Same rows, same metrics as the learned candidates in ML-08. The base rate is next to every
row. The rule's P@50 is 0.46, 0.76, 0.50 and 0.38 for February to May, and its AP is 0.199, 0.385, 0.371 and 0.234.

In [4]:
rows = []
for m, role, ev in v2_months():
    for b in BASE:
        r = ev["models"][b]
        rows.append({"month": m, "role": role, "baseline": BASE_LABEL[b], "base rate": ev["base_rate"], "P@20": r["p@20"], "P@50": r["p@50"],
                     "P@50 95% CI": r["p@50_ci"], "AP": r["ap"], "AUC": r["auc"], "top-client share of top 50": r["top_client_share_top50"]})
display(pd.DataFrame(rows))

,month,role,baseline,base rate,P@20,P@50,P@50 95% CI,AP,AUC,top-client share of top 50
0,2026-02,validation,editor's rule,0.2273,0.40,0.46,"[0.26, 0.56]",0.1985,0.3925,0.52
1,2026-02,validation,CTR above peers,0.2273,0.30,0.26,"[0.06, 0.42]",0.2335,0.5194,0.90
2,2026-02,validation,client shock,0.2273,0.30,0.34,"[0.1, 0.38]",0.2101,0.4387,0.74
3,2026-02,validation,persistence,0.2273,0.45,0.28,"[0.12, 0.5]",0.2055,0.4670,0.30
4,2026-03,validation,editor's rule,0.3382,0.60,0.76,"[0.36, 0.86]",0.3851,0.5005,0.58
5,2026-03,validation,CTR above peers,0.3382,0.40,0.38,"[0.16, 0.58]",0.3171,0.4636,0.32
6,2026-03,validation,client shock,0.3382,0.55,0.32,"[0.12, 0.7605]",0.3249,0.4746,0.60
7,2026-03,validation,persistence,0.3382,0.10,0.22,"[0.06, 0.42]",0.3070,0.4780,0.34
8,2026-04,confirmation,editor's rule,0.3079,0.50,0.50,"[0.3, 0.68]",0.3707,0.5519,0.50
9,2026-04,confirmation,CTR above peers,0.3079,0.25,0.30,"[0.16, 0.5]",0.3115,0.4977,0.30


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Question:** do the rule's top 20 make sense to a person? **Why it matters (FlyRank principle: a ranking is only useful
if its top rows are explainable):** for each of the rule's top 20 pages at the July 2026 decision point, I record the
action, the reason code, why the rule picked it, and what would make the pick wrong. Pages are shown by rank, never by ID.
The review is the V1 artefact (`baseline_top20_review.json`), still valid because the rule and the two features it reads are unchanged.

In [5]:
top20 = pd.read_json(os.path.join(OUT, "baseline_top20_review.json"))
display(top20)
print("reason codes:", top20.reason_code.value_counts().to_dict(), "| rows sharing identical position, CTR change and clicks with another row:",
      int(top20.duplicated(["position", "ctr_change_pct", "clicks_last_month"], keep=False).sum()))

,rank,position,ctr_change_pct,ctr_vs_peers_pct,clicks_last_month,reason_code,action,why,what_would_make_it_wrong
0,1,2.4,-99,118,366,TOP3_SLIPPING,review now,"position 2.4, CTR -99% last month","a tracking gap or demand drop, not AI answers ..."
1,2,1.8,-99,94,46,TOP3_SLIPPING,review now,"position 1.8, CTR -99% last month","a tracking gap or demand drop, not AI answers ..."
2,3,2.2,-99,60,57,TOP3_SLIPPING,review now,"position 2.2, CTR -99% last month","a tracking gap or demand drop, not AI answers ..."
3,4,2.7,-99,63,36,TOP3_SLIPPING,review now,"position 2.7, CTR -99% last month","a tracking gap or demand drop, not AI answers ..."
4,5,0.5,-97,222,76,TOP3_SLIPPING,review now,"position 0.5, CTR -97% last month","a tracking gap or demand drop, not AI answers ..."
5,6,3.2,-99,32,75,TOP3_SLIPPING,review now,"position 3.2, CTR -99% last month","a tracking gap or demand drop, not AI answers ..."
6,7,2.8,-98,189,87,TOP3_SLIPPING,review now,"position 2.8, CTR -98% last month","a tracking gap or demand drop, not AI answers ..."
7,8,1.7,-98,87,35,TOP3_SLIPPING,review now,"position 1.7, CTR -98% last month","a tracking gap or demand drop, not AI answers ..."
8,9,2.6,-99,93,97,TOP3_SLIPPING,review now,"position 2.6, CTR -99% last month","a tracking gap or demand drop, not AI answers ..."
9,10,3.4,-99,72,21,TOP3_SLIPPING,review now,"position 3.4, CTR -99% last month","a tracking gap or demand drop, not AI answers ..."


reason codes: {'TOP3_SLIPPING': 20} | rows sharing identical position, CTR change and clicks with another row: 3


**What the review shows.** The rule's top 20 are all top-3 pages (positions 0.5 to 3.4) whose CTR fell 94% to 99% in one
month, and every one earns more than its client's peers at that position. A one-month collapse of that size is the kind
of change that tracking gaps or demand drops produce, so the last column sends the editor to impressions and season
before anything else. That is the failure pattern the model needs to avoid.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Question:** which picks look wrong, and did any product flag or future window leak in? **Why it matters (FlyRank
principle: attack your own baseline):** a baseline that is quietly weak or leaky makes any model look better than it is.

**Weak picks.**
- **Likely tracking or demand events.** All 20 reviewed picks carry the note "a tracking gap or demand drop, not AI
  answers", because CTR fell by more than 50% in one month.
- **Repeated rows.** The last three of the 20 share identical values (position, CTR change, and last-month clicks), so
  I would review them together, not as three independent finds.
- **A tie problem.** In three of four evaluated months all 50 picks share one score, because the CTR-fall term is capped
  and many top-3 pages hit the cap (their CTR collapsed). The hash tiebreak then picks 50 of them, effectively at
  random, so the rule's precision is the erosion rate of that tied group. I keep the rule frozen as written, because
  changing it after seeing results would defeat its purpose. The distinct-score column below shows it.

In [6]:
display(pd.DataFrame(receipt("baseline_metrics.json")["per_origin"]).T)

,base_rate,p@20,p@50,p@100,ap,auc,distinct_scores_in_top50
2026-02,0.2273,0.40,0.46,0.41,0.1985,0.3925,3.0
2026-03,0.3382,0.60,0.76,0.70,0.3851,0.5005,1.0
2026-04,0.3079,0.50,0.50,0.57,0.3707,0.5519,1.0
2026-05,0.2397,0.35,0.38,0.35,0.2342,0.4526,1.0


**What it means.** The rule ranks above the base rate at 50 pages in all four months (0.38 to 0.76), but its whole-ranking quality
is near or below the base rate in two of four months, and its AUC is 0.453 on the test month, worse than random order. This
is the baseline the model has to beat. It is also the strongest baseline on AP in 3 of the 4 months, so the comparison is not
against a strawman.

**Leakage check.** The rule reads only two features, the impression-weighted position and the log CTR change between
the two closed feature months. Neither contains an outcome-window value, a product flag (the release has none) or a
future month.

In [7]:
feats = {f["name"]: f["meaning"] for f in v2("features")["features"]}
print({f: feats[f] for f in ["pos_f", "ctr_growth"]})
print("outcome-window or flag columns among the rule's inputs: none (the rule uses only the two features above)")

{'pos_f': 'impression-weighted position, feature window', 'ctr_growth': 'log CTR month -1 vs month -2 (clicks growth minus impressions growth)'}
outcome-window or flag columns among the rule's inputs: none (the rule uses only the two features above)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.